In [ ]:
from dotenv import load_dotenv
from anthropic import Anthropic
from DateTimeTool import get_current_datetime_schema, get_current_datetime

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5-20251001"

def add_user_message(messages, text):
    messages.append({"role": "user", "content": text})

def add_assistant_message(messages, text):
    messages.append({"role": "assistant", "content": text})

def chat(messages, system_prompt=None, temperature=None, toolSchemas=None):
    params = {
        "model": model,
        "max_tokens": 200,
        "messages": messages,
    }
    if system_prompt:
        params["system"] = system_prompt
    if temperature is not None:
        params["temperature"] = temperature
    if toolSchemas:
        params["tools"] = toolSchemas
    return client.messages.create(**params)

def UseTool(ToolName, Input):
    if ToolName == "get_current_datetime":
        return get_current_datetime(**Input)


def conversation(messages, continueConversation):

    while continueConversation:

        add_user_message(messages, input())

        response = chat(messages, toolSchemas=[get_current_datetime_schema])

        if response.content[0].type == "tool_use":
            ToolCallResult = UseTool(response.content[0].name, response.content[0].input)
            messages.append({"role": "assistant", "content": response.content})
            messages.append({
                "role": "user",
                "content": [{
                    "type": "tool_result",
                    "tool_use_id": response.content[0].id,
                    "content": ToolCallResult
                }]
            })
        elif response.content[0].type == "text":
            print(response.content[0].text)


In [ ]:
messages = []
add_user_message(messages, input())

response = chat(messages, toolSchemas=[get_current_datetime_schema])
response.content